# Lab 15-06 · Root Locus

Four parts, each built on `rlocus.py`. **Predict first**, then run.

In [1]:
import math
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from rlocus import locus, asymptotes, real_axis_segments, breakpoints, departure_angle, gain_at, zeta_point

## Part 1 · The rules against the numerical locus

*Predict:* how many branches cross into the right half-plane in each case?

In [2]:
cases = {
    "K/[s(s+4)]": ([1], [1, 4, 0]),
    "K/[s(s+2)(s+4)]": ([1], [1, 6, 8, 0]),
    "K(s+1)/[s(s+2)(s+4)]": ([1, 1], [1, 6, 8, 0]),
    "K(s+2)/(s^2+2s+5)": ([1, 2], [1, 2, 5]),
}
fig, axs = plt.subplots(1, 4, figsize=(13, 3.4))
Ks = np.concatenate([np.linspace(0, 1, 300), np.logspace(0, 3, 6000)])
for ax, (name, (N, D)) in zip(axs, cases.items()):
    R = locus(N, D, Ks)
    for b in range(R.shape[1]):
        ax.plot(R[:, b].real, R[:, b].imag, lw=1.2)
    ang, cen = asymptotes(N, D)
    print(f"{name:22s} asymptotes {ang} centroid {cen}  breakpoints {[(round(x, 4), round(k, 4)) for x, k in breakpoints(N, D)]}")
    print(f"{'':22s} real-axis segments {real_axis_segments(N, D, lo=-50)};  RHP at K = 1000: {int(np.sum(R[-1].real > 0))}")
    ax.axvline(0, color="gray", lw=0.5); ax.axhline(0, color="gray", lw=0.5)
    ax.set_xlim(-8, 3); ax.set_ylim(-8, 8); ax.set_title(name, fontsize=8)
plt.show()
print("departure angle from -1+j2:", round(departure_angle([1, 2], [1, 2, 5], complex(-1, 2)), 2), "deg")

K/[s(s+4)]             asymptotes [90.0, 270.0] centroid -2.0  breakpoints [(np.float64(-2.0), np.float64(4.0))]
                       real-axis segments [(np.float64(-4.0), np.float64(0.0))];  RHP at K = 1000: 0


K/[s(s+2)(s+4)]        asymptotes [60.0, 180.0, 300.0] centroid -2.0  breakpoints [(np.float64(-0.8453), np.float64(3.0792))]
                       real-axis segments [(np.float64(-2.0), np.float64(0.0)), (-50, np.float64(-4.0))];  RHP at K = 1000: 2


K(s+1)/[s(s+2)(s+4)]   asymptotes [90.0, 270.0] centroid -2.5  breakpoints [(np.float64(-2.9108), np.float64(1.5112))]
                       real-axis segments [(np.float64(-1.0), np.float64(0.0)), (np.float64(-4.0), np.float64(-2.0))];  RHP at K = 1000: 0


K(s+2)/(s^2+2s+5)      asymptotes [180.0] centroid 0.0  breakpoints [(np.float64(-4.2361), np.float64(6.4721))]
                       real-axis segments [(-50, np.float64(-2.0))];  RHP at K = 1000: 0
departure angle from -1+j2: 153.43 deg


## Part 2 · Moving a zero

*Predict:* for which $z$ can a large gain destabilize $K(s+z)/[s(s+2)(s+4)]$?

In [3]:
for z in (0.5, 1, 2, 4, 6, 8, 10):
    cen = (-6 + z) / 2
    Ks = np.logspace(-2, 4, 3000)
    unstable = [K for K in Ks if np.max(np.roots([1, 6, 8 + K, K * z]).real) > 0]
    lim = f"unstable above K = {unstable[0]:.2f} (Routh: {48 / (z - 6):.2f})" if unstable else "stable for every K"
    print(f"z = {z:4}: centroid {cen:5.2f};  {lim}")

z =  0.5: centroid -2.75;  stable for every K
z =    1: centroid -2.50;  stable for every K


z =    2: centroid -2.00;  stable for every K


z =    4: centroid -1.00;  stable for every K
z =    6: centroid  0.00;  stable for every K


z =    8: centroid  1.00;  unstable above K = 24.05 (Routh: 24.00)


z =   10: centroid  2.00;  unstable above K = 12.05 (Routh: 12.00)


## Part 3 · A PLL across its gain

In [4]:
tz, tp = 5e-6, 5e-6 / 11
Ks = np.logspace(9.5, 12, 1500)
zeta = []
for K in Ks:
    r = np.roots([tp, 1, K * tz, K])
    c = r[np.abs(r.imag) > 1e-6 * np.abs(r)]
    zeta.append(-c[0].real / abs(c[0]) if len(c) else 1.0)
fig, ax = plt.subplots(figsize=(6, 3.2))
ax.semilogx(Ks, zeta)
for x, k in breakpoints([tz, 1], [tp, 1, 0, 0]):
    ax.axvline(k, color="gray", ls=":")
ax.axvline(1e11 / 1.1, color="red", ls="--", label="design K")
ax.set_xlabel("K (1/s²)"); ax.set_ylabel("ζ of the complex pair"); ax.legend(); ax.grid(True, alpha=0.3)
plt.show()
for K in (5e10, 1e11 / 1.1, 2e11):
    r = np.roots([tp, 1, K * tz, K]); c = r[np.argmax(r.imag)]
    print(f"K = {K:.3e}: complex pair {c:.4g}, zeta = {-c.real / abs(c):.3f}")

K = 5.000e+10: complex pair -1.268e+05+2.011e+05j, zeta = 0.533
K = 9.091e+10: complex pair -2.63e+05+2.243e+05j, zeta = 0.761
K = 2.000e+11: complex pair -9.703e+05+8.69e+05j, zeta = 0.745


## Part 4 · The PD servo in ngspice

In [5]:
import sys
sys.path.insert(0, "../15-03-lab")
from specs import step_specs
r = step_specs([4, 32], [1, 8, 32], 5)
print(f"Python: overshoot {100 * r['overshoot']:.2f}%, settling {r['settle']:.3f} s")
if shutil.which("ngspice"):
    out = subprocess.run(["ngspice", "-b", "pd_servo.cir"], capture_output=True, text=True, timeout=120).stdout
    print("ngspice:", [l for l in out.splitlines() if l.split("=")[0].strip() in ("os", "ts")])
else:
    print("ngspice not installed: skip (CI runs pd_servo.cir through tools/run_spice.py)")

Python: overshoot 6.70%, settling 0.932 s


ngspice: ['os = 6.702000e+00', 'ts = 9.326836e-01']


**Discussion.** Answer the four questions in the [lab handout](README.md) before reading its answers.